<a href="https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/main/notebooks/demo_gcad_e_prime_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# E-Prime Logits Constraint Retention In Colab

This notebook copies the GCAD E-Prime long-conversation setup, but adds a decoding-time E-Prime logits processor. The processor blocks next-token candidates when appending them would create a state-of-being verb or a listed contraction in the assistant output.

It can compare prompt-only, GCAD, logits constraint, and GCAD plus logits constraint conditions.


### Installation

Run this setup cell once in a fresh Colab GPU runtime before continuing. It clones the repo and installs the CUDA-compatible notebook dependencies.


In [ ]:
!rm -f /content/gcad_eprime_vectors.pt
import os
os.environ["GCAD_BOOTSTRAP_FROM_HIDDEN_STATES"] = "1"


In [ ]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
#
# PURPOSE:
# This cell prepares the environment to run vLLM and its custom plugins.
# It handles complex dependency conflicts common in Colab (e.g., pre-installed
# torch versions) and ensures the plugin code is loaded correctly.
#
# KEY ACTIONS:
# 1. Clones the vLLM-Hook repository if not present.
# 2. Installs a compatible version of vLLM and PyTorch for your GPU.
# 3. Cleans up old binary artifacts to prevent version conflicts.
# 4. Loads the plugin source code.
# 5. TRIGGERS A COLAB RESTART: The cell will restart the runtime to ensure
#    the new libraries are fully loaded into the kernel memory.
#
# EXPECTED BEHAVIOR:
# - The cell will run and install packages.
# - It may print "Restarting Colab runtime...".
# - The cell will stop abruptly, and the runtime will restart.
# - The restart is automatic; the code will resume from the top.
# ==============================================================================

from pathlib import Path
import importlib
import importlib.util
import os
import re
import shutil
import site
import subprocess
import sys
import time

# Configuration
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "codex/MLR-42-gcad-eprime-worker")
REPO_NAME = "vLLM-Hook"

# Environment Variables (Optional overrides)
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.11,<0.19")
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128")

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()

# --------------------------------------------------------------------------
# Helper Functions
# --------------------------------------------------------------------------

def run(cmd, cwd=None, env=None):
    """
    Executes a shell command, printing output in real-time.
    Raises an error if the command fails.
    """
    cmd = [str(part) for part in cmd]
    print(f"> Running: {' '.join(cmd)}", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail = tail[-120:]

    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )

def run_capture(cmd):
    """Executes a command and returns stdout/stderr without printing."""
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)

def norm(name):
    """Normalize package names for comparison."""
    return name.lower().replace("_", "-")

def package_from_req_line(line: str) -> str:
    """Extract package name from a requirement string (e.g., 'torch>=1.0' -> 'torch')."""
    stripped = line.strip()
    # Split on version specifiers
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())

def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    """Checks if the git repo's origin matches the expected URL."""
    try:
        url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return url == expected_remote

def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    """Searches up the directory tree for a matching git repo."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None

def assert_cuda_runtime():
    """Ensures a GPU is available before proceeding."""
    try:
        import torch
    except Exception:
        torch = None

    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None

    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "No CUDA GPU detected. "
            "In Colab, go to Runtime > Change runtime type and select T4 GPU (or better), "
            "then re-run the entire notebook from the beginning."
        )

# --------------------------------------------------------------------------
# 1. Repository Setup
# --------------------------------------------------------------------------

expected_remote = REPO_URL.removesuffix(".git")
existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)

if IN_COLAB:
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Refreshing existing repo at: {REPO_ROOT}")
        run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch detected. Replacing clone...")
            shutil.rmtree(REPO_ROOT)
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        else:
            print(f"Refreshing existing clone...")
            run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Working directory set to: {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
FILTERED_REQ_FILE = Path("/tmp/vllm_hook_colab_requirements.txt")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_binary_deps_restarted")

print(f"\n--- Environment Summary ---")
print(f"Running in Colab: {IN_COLAB}")
print(f"Repo Root: {REPO_ROOT}")
print(f"Plugin Dir: {PKG_DIR}")

if IN_COLAB:
    assert_cuda_runtime()

# --------------------------------------------------------------------------
# 2. Plugin Directory Validation
# --------------------------------------------------------------------------

if not PKG_DIR.exists():
    raise FileNotFoundError(
        f"Plugin directory not found at {PKG_DIR}. "
        "Please ensure the repository was cloned correctly."
    )

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("git is required but unavailable in this runtime.")

# --------------------------------------------------------------------------
# 3. Dependency Management
# --------------------------------------------------------------------------

if REQ_FILE.exists():
    keep = []
    # These packages are managed by Colab's pre-installed environment.
    # Installing them again can cause conflicts.
    blocked = {"vllm", "torch", "torchvision", "torchaudio", "numpy", "scipy", "protobuf"}

    for line in REQ_FILE.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue

        package = package_from_req_line(stripped)
        if package in blocked:
            print(f"⏭ Skipping managed dependency: {line}")
            continue
        keep.append(line)

    FILTERED_REQ_FILE.write_text("\n".join(keep) + "\n")
    print(f"Installing filtered requirements from {REQ_FILE.name}...")
    run([sys.executable, "-m", "pip", "install", "-r", str(FILTERED_REQ_FILE)])
else:
    print("⚠ Warning: No requirement.txt found; skipping custom dependency installation.")

# Ensure protobuf is at the correct version
print("Ensuring protobuf version compatibility...")
run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])

# --------------------------------------------------------------------------
# 4. vLLM & PyTorch Installation
# --------------------------------------------------------------------------

if COLAB_INSTALL_VLLM:
    print(f"Installing user-specified vLLM: {COLAB_INSTALL_VLLM}")
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    print(f"\nInstalling vLLM and PyTorch for {VLLM_TORCH_BACKEND}...")

    # 1. Uninstall existing conflicting versions
    run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch", "torchvision", "torchaudio"])

    # 2. Manually remove leftover binary artifacts that pip might miss
    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        leftovers = [
            site_path / "vllm", *site_path.glob("vllm-*.dist-info"),
            site_path / "torch", *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision", *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio", *site_path.glob("torchaudio-*.dist-info"),
        ]
        for leftover in leftovers:
            if leftover.exists():
                print(f"  Cleaning up leftover: {leftover.name}")
                if leftover.is_dir():
                    shutil.rmtree(leftover)
                else:
                    leftover.unlink()

    # 3. Force clear GPU memory
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except:
        pass

    # 4. Install using 'uv' (faster than pip) with specific CUDA backend
    print("  Downloading and installing packages with 'uv'...")
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv", "pip", "install",
        "--system",
        VLLM_SPEC,
        "torch", "torchvision", "torchaudio",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

    # 5. Verify Scipy/Numpy
    scipy_check = run_capture([
        sys.executable, "-c", "import numpy, scipy; print('numpy', numpy.__version__); print('scipy', scipy.__version__)"
    ])
    if scipy_check.returncode:
        print("  Detected numpy/scipy issues; upgrading...")
        run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy", "scipy"])

# Verify Installation
print("\nVerifying installation...")
run([
    sys.executable,
    "-c",
    "import torch, vllm; print('✓ torch:', torch.__version__); print('✓ vllm:', getattr(vllm, '__version__', 'unknown'))"
])

# --------------------------------------------------------------------------
# 5. Plugin Loading
# --------------------------------------------------------------------------

# Strategy: Add path to sys.path to allow immediate import,
# then ensure metadata is installed for subprocesses later.
plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

try:
    spec = importlib.util.spec_from_file_location("vllm_hook_plugins", PKG_DIR / "__init__.py")
    if spec:
        importlib.util.module_from_spec(spec)
        print("✓ Plugin module loaded successfully from source path.")
except Exception as e:
    print(f"⚠ Warning: Initial import check failed ({e}). This is expected if compilation is needed later.")

# Ensure the package is registered in the environment (metadata)
# This is necessary for tools that rely on `importlib.metadata`.
print("Registering plugin package in environment...")
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])

print("Installing W&B tracking client...")
run([sys.executable, "-m", "pip", "install", "wandb"])

print(f"Plugin Source: {plugin_src_dir}")
print(f"Python Exec  : {sys.executable}")

# --------------------------------------------------------------------------
# 6. Runtime Restart (Critical for Colab)
# --------------------------------------------------------------------------
# We restart the runtime to ensure the newly installed binary libraries
# are loaded into a fresh Python interpreter. This prevents "dirty state" issues.
if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("\n" + "="*50)
    print("RESTARTING COLAB RUNTIME...")
    print("This ensures the new vLLM/Torch binaries are fully loaded.")
    print("Do not interrupt this process.")
    print("="*50)

    time.sleep(1) # Allow output buffer to flush
    sys.exit(0) # Terminate this process to trigger Colab restart


### Scoring Dependencies

The E-Prime checker uses spaCy for state-of-being verb detection. Run this cell before scoring.


In [ ]:
# Optional scoring dependency setup for Colab.
# Run after the main setup cell. If you already have spaCy and en_core_web_sm, this is a no-op.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("spacy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])

try:
    import spacy
    spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


### Imports & Environment


In [ ]:
import copy
import gc
import io
import json
import os
import multiprocessing as mp
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from vllm import SamplingParams

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from vllm_hook_plugins import HookLLM, register_plugins
from vllm_hook_plugins.logits_processors import default_eprime_bad_words
from research.experiment_tracking import (
    extract_current_assistant_response,
    ExperimentCondition,
    ExperimentResult,
    RunMetrics,
    TurnResult,
    WandbTracker,
    evaluate_mlr20_condition,
    log_mlr20_results_sequentially,
    mlr20_alpha_sweep_conditions,
    run_mlr20_batch,
    collect_runtime_provenance,
    format_gib,
    has_required_vram,
    required_vram_bytes,
    turn_result_from_eprime_row,
)

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


### GPU Memory Preflight


In [ ]:
def show_nvidia_smi():
    if not IN_COLAB:
        return
    try:
        completed = subprocess.run(
            ["nvidia-smi"],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        print("nvidia-smi unavailable in this runtime.")
        return
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)


def cuda_memory_preflight(gpu_memory_utilization, *, show_smi=True):
    gc.collect()
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. In Colab, select a GPU runtime, restart the runtime, "
            "and run the notebook from the beginning."
        )

    torch.cuda.empty_cache()
    if hasattr(torch.cuda, "ipc_collect"):
        try:
            torch.cuda.ipc_collect()
        except Exception as exc:
            print(f"torch.cuda.ipc_collect() skipped: {exc}")

    free_bytes, total_bytes = torch.cuda.mem_get_info()
    required_bytes = required_vram_bytes(total_bytes, gpu_memory_utilization)
    print(
        "CUDA memory after cleanup: "
        f"free={format_gib(free_bytes)} total={format_gib(total_bytes)} "
        f"required={format_gib(required_bytes)} "
        f"for gpu_memory_utilization={gpu_memory_utilization:.2f}"
    )

    if show_smi:
        show_nvidia_smi()

    if not has_required_vram(
        free_bytes=free_bytes,
        total_bytes=total_bytes,
        gpu_memory_utilization=gpu_memory_utilization,
    ):
        raise RuntimeError(
            "Not enough free CUDA memory for the configured vLLM reservation. "
            f"Free={format_gib(free_bytes)}, required={format_gib(required_bytes)}, "
            f"total={format_gib(total_bytes)}, "
            f"gpu_memory_utilization={gpu_memory_utilization:.2f}. "
            "Restart the Colab runtime and choose Runtime > Run all. "
            "Do not continue from a partially executed runtime."
        )

    return {
        "free_bytes": free_bytes,
        "total_bytes": total_bytes,
        "required_bytes": required_bytes,
    }


### Initialize `HookLLM` With GCAD


In [ ]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = float(os.environ.get("VLLM_HOOK_GPU_MEMORY_UTILIZATION", "0.30" if IN_COLAB else "0.65"))

GCAD_VECTOR_PATH = os.environ.get(
    "GCAD_VECTOR_PATH",
    "/content/gcad_eprime_vectors.pt" if IN_COLAB else "notebooks/artifacts/gcad_eprime_vectors.pt",
)
GCAD_C_BASE = float(os.environ.get("GCAD_C_BASE", "3.5"))
GCAD_SHARPNESS = float(os.environ.get("GCAD_SHARPNESS", "1.5"))
GCAD_HOOKS_ON = os.environ.get("GCAD_HOOKS_ON", "decode")

def bootstrap_gcad_vector_from_hidden_states(vector_path):
    """Create a GCAD-shaped, nonzero artifact from contrastive hidden states.

    This is an implementation smoke/ablation vector, not faithful cropped
    attention-delta extraction from the Prompt-Activation Duality paper.
    It answers: does the GCAD worker path change behavior when given a
    nonzero direction in the expected artifact format?
    """
    print("Bootstrapping GCAD-shaped vector from contrastive hidden states.")
    print("NOTE: this is not faithful cropped attention-delta extraction.")

    vector_path.parent.mkdir(parents=True, exist_ok=True)
    script_path = vector_path.with_suffix(".bootstrap.py")
    script_path.write_text(
        f'''
from pathlib import Path
import gc
import os
import sys

import torch
from transformers import AutoConfig
from vllm import SamplingParams

repo_root = {str(REPO_ROOT)!r}
if repo_root not in sys.path:
    sys.path.append(repo_root)

os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = {os.environ.get("VLLM_WORKER_MULTIPROC_METHOD", "fork")!r}
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = {os.environ.get("VLLM_ENABLE_V1_MULTIPROCESSING", "0")!r}

from vllm_hook_plugins import HookLLM, register_plugins

register_plugins()

model = {model!r}
cache_dir = {cache_dir!r}
vector_path = Path({str(vector_path)!r})
gpu_memory_utilization = {GPU_MEMORY_UTILIZATION!r}
max_model_len = {MAX_MODEL_LEN!r}
c_base = {GCAD_C_BASE!r}
sharpness = {GCAD_SHARPNESS!r}

hf_cfg = AutoConfig.from_pretrained(model, trust_remote_code=True, cache_dir=cache_dir)
text_cfg = getattr(hf_cfg, "text_config", hf_cfg)
hidden_size = int(getattr(text_cfg, "hidden_size"))
num_heads = int(getattr(text_cfg, "num_attention_heads"))
head_dim = hidden_size // num_heads

extractor = HookLLM(
    model=model,
    worker_name="probe_hidden_states",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=gpu_memory_utilization,
    max_model_len=max_model_len,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

positive_prompt = """System instruction: Write the answer in strict E-Prime. Avoid all forms of to be, including am, is, are, was, were, be, being, and been.

Question: Explain why instruction-following can drift over a long conversation.
Answer:"""
negative_prompt = """System instruction: Write a clear helpful answer.

Question: Explain why instruction-following can drift over a long conversation.
Answer:"""
probe_params = SamplingParams(temperature=0.0, max_tokens=1)

pos = extractor.generate([positive_prompt], sampling_params=probe_params, use_hook=True)[0]
neg = extractor.generate([negative_prompt], sampling_params=probe_params, use_hook=True)[0]
pos_cache = pos.probes["hs_cache"]
neg_cache = neg.probes["hs_cache"]

layers = {{}}
for module_name, pos_entry in pos_cache.items():
    neg_entry = neg_cache.get(module_name)
    if neg_entry is None:
        continue
    layer_num = int(pos_entry["layer_num"])
    pos_h = torch.as_tensor(pos_entry["hidden_states"]).float().reshape(-1, hidden_size).mean(dim=0)
    neg_h = torch.as_tensor(neg_entry["hidden_states"]).float().reshape(-1, hidden_size).mean(dim=0)
    delta = pos_h - neg_h
    norm = delta.norm()
    if norm > 0:
        delta = delta / norm
    layers[layer_num] = {{
        "delta": delta.cpu(),
        "mean_sys_key": torch.zeros(num_heads, head_dim),
        "d_bar": torch.tensor(0.0),
        "c_base": c_base,
        "sharpness": sharpness,
        "source_module": module_name,
        "extraction_method": "contrastive_hidden_state_bootstrap",
    }}

extractor.close()
del extractor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if not layers:
    raise ValueError("Hidden-state bootstrap did not capture any layers")

torch.save(
    {{
        "metadata": {{
            "model": model,
            "method": "contrastive_hidden_state_bootstrap",
            "warning": "GCAD-shaped artifact only; not faithful cropped attention-delta extraction",
            "positive_prompt": positive_prompt,
            "negative_prompt": negative_prompt,
        }},
        "layers": layers,
    }},
    vector_path,
)
print(f"Saved bootstrap GCAD-shaped vector to {{vector_path}}")
''',
        encoding="utf-8",
    )
    subprocess.run([sys.executable, str(script_path)], check=True)
    print(f"Saved bootstrap GCAD-shaped vector to {vector_path}")

vector_path = Path(GCAD_VECTOR_PATH)
if (
    not vector_path.exists()
    and os.environ.get("GCAD_BOOTSTRAP_FROM_HIDDEN_STATES", "0") == "1"
):
    bootstrap_gcad_vector_from_hidden_states(vector_path)

if not vector_path.exists():
    if os.environ.get("GCAD_ALLOW_ZERO_VECTOR_SMOKE", "0") == "1":
        vector_path.parent.mkdir(parents=True, exist_ok=True)
        hidden_size = 1536
        num_heads = 12
        head_dim = hidden_size // num_heads
        torch.save(
            {
                "metadata": {
                    "purpose": "GCAD worker smoke test only; do not use for behavioral claims",
                    "model": model,
                },
                "layers": {
                    12: {
                        "delta": torch.zeros(hidden_size),
                        "mean_sys_key": torch.zeros(num_heads, head_dim),
                        "d_bar": torch.tensor(0.0),
                        "c_base": GCAD_C_BASE,
                        "sharpness": GCAD_SHARPNESS,
                    }
                },
            },
            vector_path,
        )
        print(f"Created zero GCAD smoke-test vector at {vector_path}")
    else:
        raise FileNotFoundError(
            f"GCAD vector artifact not found at {vector_path}. "
            "Set GCAD_VECTOR_PATH to a real extracted vector artifact, or set "
            "GCAD_ALLOW_ZERO_VECTOR_SMOKE=1 only to verify notebook wiring."
        )

gcad_artifact = torch.load(vector_path, map_location="cpu", weights_only=False)
gcad_layers = gcad_artifact.get("layers", {})
if not gcad_layers:
    raise ValueError(f"GCAD artifact at {vector_path} has no layers")

layer_diagnostics = []
for layer_key, entry in gcad_layers.items():
    delta = torch.as_tensor(entry["delta"])
    mean_sys_key = torch.as_tensor(entry["mean_sys_key"])
    layer_diagnostics.append(
        {
            "layer": int(layer_key),
            "delta_norm": float(delta.norm().item()),
            "mean_sys_key_norm": float(mean_sys_key.norm().item()),
            "d_bar": float(torch.as_tensor(entry.get("d_bar", 0.0)).item()),
        }
    )

print("GCAD vector diagnostics:")
for row in layer_diagnostics:
    print(row)

if all(row["delta_norm"] == 0.0 for row in layer_diagnostics):
    if os.environ.get("GCAD_ALLOW_ZERO_VECTOR_SMOKE", "0") == "1":
        print("WARNING: GCAD vector has zero delta norm; this is smoke-test only and should match baseline.")
    else:
        raise ValueError(
            f"GCAD vector artifact at {vector_path} has zero delta norm for every layer. "
            "Delete the smoke-test vector or set GCAD_VECTOR_PATH to a real extracted artifact."
        )

gpu_memory_preflight = cuda_memory_preflight(GPU_MEMORY_UTILIZATION)

llm = HookLLM(
    model=model,
    worker_name="probe_gcad",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

print(f"Model loaded: {model}")
print("GCAD worker enabled")
print(f"GCAD vector path: {vector_path}")
print(f"GCAD c_base={GCAD_C_BASE} sharpness={GCAD_SHARPNESS} hooks_on={GCAD_HOOKS_ON}")
print(f"Max model length: {MAX_MODEL_LEN}")
print(f"GPU memory utilization: {GPU_MEMORY_UTILIZATION}")


### E-Prime Long-Conversation Configuration


In [ ]:
# E_PRIME_CONSTRAINT = """Persistent E-Prime constraint:
# Exclude state-of-being verbs (am, is, are, was, were, be, being, been) in your responses.
# Exclude contractions containing state-of-being verbs, such as I'm, you're, we're, they're, he's, she's, it's, that's, there's, etc. in your responses.
# """.strip()


E_PRIME_CONSTRAINT ="""
    Prefer action verbs and relational phrasing over state of being verbs.
    Prefer forms like functions as, appears as, counts as, refers to,
    describes, and acts through.
"""



SYSTEM_MESSAGE = (
    "You are a concise assistant continuing a long Talk2AI-style conversation. "
    "Preserve the E-Prime constraint throughout the whole conversation while still answering helpfully."
)

SEED_HISTORY = [
    {"role": "user", "content": "I keep hearing that climate change discussions are exaggerated compared with other problems."},
    {"role": "assistant", "content": "Let's compare long-run measurements with the way headlines describe them."},
    {"role": "user", "content": "The headlines make me distrust the whole topic."},
    {"role": "assistant", "content": "Repeated alarm can make even strong evidence feel performative."},
]

USER_TURNS = [
    "What evidence should I look at if I want to avoid headline-driven conclusions?",
    "How do local weather experiences confuse the broader trend?",
    "What tradeoffs matter for households when policies raise energy costs?",
    "Where does adaptation make sense, and where does it fall short?",
    "How should poorer countries think about growth and emissions limits?",
    "What practical local policy question should I ask a city council candidate?",
    "How can I separate serious policy criticism from misinformation?",
    "What should climate communicators stop doing if they want skeptical people to listen?",
    "What should good-faith skeptics concede before debating policy?",
    "End with a cautious, practical position that still respects the evidence.",
]

ALPHA = 0.2
SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=160)
USE_EPRIME_LOGITS_PROCESSOR = os.environ.get("USE_EPRIME_LOGITS_PROCESSOR", "1") == "1"
MAX_USER_TURNS = min(10, len(USER_TURNS))
HISTORY_WINDOW_MESSAGES = 16


### Prompt And Run Helpers


In [ ]:
def render_e_prime_prompt(history, user_message):
    recent_history = history[-HISTORY_WINDOW_MESSAGES:]
    omitted_messages = max(0, len(history) - len(recent_history))
    transcript = "\n".join(
        f"{item['role'].upper()}: {item['content']}" for item in recent_history
    )
    if transcript:
        transcript += "\n"

    earlier_context = ""
    if omitted_messages:
        earlier_context = (
            f"Earlier conversation context: {omitted_messages} older messages are omitted "
            "from this prompt to keep the Colab run within memory limits. Continue the same conversation.\n\n"
        )

    return f"""{SYSTEM_MESSAGE}

{E_PRIME_CONSTRAINT}

{earlier_context}Recent conversation:
{transcript}USER: {user_message}
ASSISTANT:""".strip()



STATE_OF_BEING_CORE_WORDS = {"am", "is", "are", "was", "were", "be", "being", "been"}
STATE_OF_BEING_SPACY_MODEL = "en_core_web_sm"
CONTRACTION_PATTERN = re.compile(
    r"\b(?:i'm|you're|we're|they're|he's|she's|it's|that's|there's|what's|who's|where's|when's|why's|how's)\b",
    re.IGNORECASE,
)


def load_state_of_being_nlp(model_name=STATE_OF_BEING_SPACY_MODEL):
    try:
        import spacy
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "spaCy is required for the E-Prime checker. Run the scoring dependency setup cell first."
        ) from exc

    try:
        return spacy.load(model_name, disable=["ner", "parser"])
    except OSError as exc:
        raise OSError(
            f"spaCy model {model_name!r} is not installed. Run the scoring dependency setup cell first."
        ) from exc


state_of_being_nlp = load_state_of_being_nlp()


def is_state_of_being_token(token):
    lemma = token.lemma_.lower()
    text = token.text.lower()
    if lemma == "be" and token.pos_ in {"AUX", "VERB"}:
        return True
    return text in STATE_OF_BEING_CORE_WORDS and token.pos_ in {"AUX", "VERB"}


def check_e_prime_violations(text, nlp=state_of_being_nlp):
    text = text or ""
    doc = nlp(text)
    verb_matches = []
    for token in doc:
        if is_state_of_being_token(token):
            verb_matches.append(
                {
                    "text": token.text,
                    "lemma": token.lemma_,
                    "pos": token.pos_,
                    "tag": token.tag_,
                    "start": token.idx,
                    "end": token.idx + len(token.text),
                }
            )
    contraction_matches = [
        {"text": match.group(0), "start": match.start(), "end": match.end()}
        for match in CONTRACTION_PATTERN.finditer(text)
    ]
    violation_count = len(verb_matches) + len(contraction_matches)
    return {
        "state_of_being_count": len(verb_matches),
        "contraction_count": len(contraction_matches),
        "e_prime_violation_count": violation_count,
        "e_prime_retained": violation_count == 0,
        "e_prime_score": 1.0 if violation_count == 0 else 0.0,
        "state_of_being_matches": verb_matches,
        "contraction_matches": contraction_matches,
    }


def rows_from_experiment_results(experiment_results):
    rows = []
    for experiment_result in experiment_results:
        rows.extend(experiment_result.full_result["rows"])
    return rows


In [ ]:
def sampling_params_with_eprime_constraint(base_params=SAMPLING_PARAMS):
    """Return SamplingParams using vLLM's native bad_words processor."""
    return SamplingParams(
        temperature=base_params.temperature,
        max_tokens=base_params.max_tokens,
        bad_words=default_eprime_bad_words(),
    )


EPRIME_CONSTRAINED_SAMPLING_PARAMS = sampling_params_with_eprime_constraint()
print("E-Prime bad_words logits processor attached")


### Run Prompt, GCAD, Logits, And Combined Conditions


In [ ]:
def generate_with_gcad(llm, prompts, sampling_params=None, use_gcad=True, **kwargs):
    if isinstance(prompts, str):
        prompts = [prompts]

    base_params = sampling_params or SamplingParams(**kwargs)
    sp_list = []
    for _ in prompts:
        sp = copy.copy(base_params)
        extra = dict(sp.extra_args or {})
        if use_gcad:
            extra["gcad"] = {
                "vector_path": str(vector_path),
                "c_base": GCAD_C_BASE,
                "sharpness": GCAD_SHARPNESS,
                "hooks_on": GCAD_HOOKS_ON,
            }
        sp.extra_args = extra
        sp_list.append(sp)

    return llm.generate(
        prompts=prompts,
        sampling_params=sp_list,
        use_hook=use_gcad,
    )


def run_gcad_long_conversation_condition(condition_name, use_gcad, sampling_params):
    history = list(SEED_HISTORY)
    rows = []
    started = time.perf_counter()
    for turn_index, user_message in enumerate(USER_TURNS, start=1):
        prompt = render_e_prime_prompt(history, user_message)
        outputs = generate_with_gcad(
            llm,
            prompt,
            sampling_params=sampling_params,
            use_gcad=use_gcad,
        )
        reply = extract_current_assistant_response(outputs[0].outputs[0].text, prompt)
        score = check_e_prime_violations(reply)
        rows.append(
            {
                "condition": condition_name,
                "turn": turn_index,
                "user_message": user_message,
                "reply": reply,
                "prompt_characters": len(prompt),
                "response_characters": len(reply),
                "elapsed_s": time.perf_counter() - started,
                **score,
            }
        )
        history.extend(
            [
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": reply},
            ]
        )
    return rows

batch_started = time.perf_counter()
condition_specs = [
    ("baseline_prompt_only", False, SAMPLING_PARAMS),
    ("gcad", True, SAMPLING_PARAMS),
    ("eprime_logits", False, EPRIME_CONSTRAINED_SAMPLING_PARAMS),
    ("gcad_eprime_logits", True, EPRIME_CONSTRAINED_SAMPLING_PARAMS),
]
all_rows = []
for condition_name, use_gcad, sampling_params in condition_specs:
    all_rows.extend(
        run_gcad_long_conversation_condition(
            condition_name,
            use_gcad=use_gcad,
            sampling_params=sampling_params,
        )
    )
batch_elapsed_s = time.perf_counter() - batch_started
results = pd.DataFrame(all_rows)
print(f"Prompt + GCAD + logits batch complete in {batch_elapsed_s:.2f}s")
results


### Validate E-Prime Scoring Scope


In [ ]:
if "reply" not in results.columns:
    raise ValueError("Expected generated assistant replies in the results table.")
if not set(["e_prime_score", "e_prime_violation_count", "state_of_being_matches", "contraction_matches"]).issubset(results.columns):
    raise ValueError("Expected assistant-only E-Prime scores from the condition evaluator.")
print("E-Prime compliance was scored only on newly generated assistant replies.")
results[["condition", "turn", "e_prime_score", "e_prime_violation_count", "reply"]].head()


### Compare Constraint Retention


In [ ]:
summary = results.groupby("condition").agg(
    mean_e_prime_retention=("e_prime_score", "mean"),
    violation_rate=("e_prime_retained", lambda values: 1 - values.mean()),
    mean_violations=("e_prime_violation_count", "mean"),
    mean_state_of_being_count=("state_of_being_count", "mean"),
    mean_contraction_count=("contraction_count", "mean"),
    mean_elapsed_s=("elapsed_s", "mean"),
    max_prompt_chars=("prompt_characters", "max"),
)

progression = results.sort_values(["condition", "turn"]).copy()
progression["turn_has_violation"] = progression["e_prime_violation_count"] > 0
progression["cumulative_violations"] = progression.groupby("condition")["e_prime_violation_count"].cumsum()
progression["cumulative_violation_turns"] = progression.groupby("condition")["turn_has_violation"].cumsum()
progression["running_retention_rate"] = progression.groupby("condition")["e_prime_retained"].expanding().mean().reset_index(level=0, drop=True)

first_violation_turn = (
    progression[progression["turn_has_violation"]]
    .groupby("condition")["turn"]
    .min()
    .rename("first_violation_turn")
)
summary = summary.join(first_violation_turn)
summary["first_violation_turn"] = summary["first_violation_turn"].fillna("none")

trajectory = progression[
    [
        "condition",
        "turn",
        "e_prime_violation_count",
        "state_of_being_count",
        "contraction_count",
        "cumulative_violations",
        "cumulative_violation_turns",
        "running_retention_rate",
        "reply",
    ]
]

print("Aggregate summary")
display(summary)
print("Turn-by-turn trajectory")
display(trajectory)


### Inspect Failures


In [ ]:
failures = results[results["e_prime_score"] < 1.0][
    [
        "condition",
        "turn",
        "user_message",
        "e_prime_violation_count",
        "state_of_being_matches",
        "contraction_matches",
        "reply",
    ]
]
failures


### Save Results And Limitations


In [ ]:
out_dir = Path("/content/eprime_logits_results") if IN_COLAB else Path("notebooks/results")
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path = out_dir / f"eprime_logits_constraint_retention_{stamp}.csv"
json_path = out_dir / f"eprime_logits_constraint_retention_{stamp}.json"

results.to_csv(csv_path, index=False)
json_path.write_text(
    json.dumps(
        {
            "model": model,
            "conditions": ["baseline_prompt_only", "gcad", "eprime_logits", "gcad_eprime_logits"],
            "constraint": E_PRIME_CONSTRAINT,
            "metric": "e_prime_score == 1.0 when no state-of-being verbs or listed contractions appear",
            "max_user_turns": MAX_USER_TURNS,
            "history_window_messages": HISTORY_WINDOW_MESSAGES,
            "max_model_len": MAX_MODEL_LEN,
            "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
            "summary": summary.reset_index().to_dict(orient="records"),
            "limitations": [
                "Notebook smoke validation checks structure and platform-specific imports only unless this notebook is run in Colab.",
                "The contraction checker uses an explicit list and may not catch every informal contraction.",
                "The logits processor enforces local surface-form bans, not a full natural-language grammar.",
                "The spaCy POS-based checker can miss malformed output or mis-tag unusual phrasing.",
            ],
            "rows": results.to_dict(orient="records"),
            "gcad_vector_path": str(vector_path),
            "gcad_c_base": GCAD_C_BASE,
            "gcad_sharpness": GCAD_SHARPNESS,
            "gcad_hooks_on": GCAD_HOOKS_ON,
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Saved CSV: {csv_path}")
print(f"Saved JSON: {json_path}")
print("Validation note: this committed notebook has been smoke-validated locally; run it in a Colab GPU runtime for execution results.")


### W&B Tracking


In [ ]:
print("This GCAD notebook reuses MLR-20 data/scoring but does not log through the MLR-20 A0-A4 W&B helper.")
print("Log manually or add a new condition helper once GCAD has a Notion design-matrix row.")
